In [ ]:
import numpy as np
import mne

# --------------------------
# Settings
# --------------------------
EDF_PATH = "Participant_043_EC.edf"   # path to the single .edf file
BLOCK_DURATION_SEC = 4                # length of each analysis block
ALPHA_BAND = (8, 13)                  # full alpha band (Hz)

# --------------------------
# Load the EDF
# --------------------------
raw = mne.io.read_raw_edf(EDF_PATH, preload=True, verbose=False)
print(f"Loaded {EDF_PATH}: channels = {raw.ch_names}, "
      f"sfreq = {raw.info['sfreq']} Hz, duration = {raw.times[-1]:.1f} s")

# --------------------------
# Re-reference AF7/AF8 to the average of TP9/TP10 (as in the original notebook)
# --------------------------
ref_data = raw.get_data(picks=["TP9", "TP10"]).mean(axis=0)
for ch in ["AF7", "AF8"]:
    raw._data[raw.ch_names.index(ch), :] -= ref_data

# --------------------------
# Split into 4-second blocks
# --------------------------
events = mne.make_fixed_length_events(raw, duration=BLOCK_DURATION_SEC)
epochs = mne.Epochs(raw, events, tmin=0, tmax=BLOCK_DURATION_SEC - 1 / raw.info["sfreq"],
                    baseline=None, preload=True, verbose=False)
print(f"Number of {BLOCK_DURATION_SEC}-s blocks: {len(epochs)}")

# --------------------------
# FAA (full alpha, power difference in dB) per block:
#   FAA = 10*log10(AF8 alpha power) - 10*log10(AF7 alpha power)
# --------------------------
faa_per_block = []
for i in range(len(epochs)):
    block = mne.io.RawArray(epochs[i].get_data()[0], epochs.info, verbose=False)
    psd = block.compute_psd(fmin=1, fmax=45, method="welch", n_fft=1024,
                            picks=["AF7", "AF8"], verbose=False)
    psds, freqs = psd.get_data(return_freqs=True)
    band = (freqs >= ALPHA_BAND[0]) & (freqs <= ALPHA_BAND[1])
    af7_power, af8_power = psds[0, band].mean(), psds[1, band].mean()
    faa = 10 * np.log10(af8_power) - 10 * np.log10(af7_power)
    faa_per_block.append(faa)
    print(f"Block {i+1:2d}: AF7 = {af7_power:.4e}, AF8 = {af8_power:.4e}, FAA = {faa:.4f} dB")

# --------------------------
# Result
# --------------------------
print(f"\nMean FAA (power difference, dB, full alpha 8-13 Hz) across {len(faa_per_block)} blocks: "
      f"{np.mean(faa_per_block):.4f}")

: 